# Benchmark A — fine-tuning on KaggleProduces the task vectors for *Is Mergeability Algorithm-Relative?***Why here and not locally.** The project was developed on a fanless MacBook Air M4. The samefine-tuning job takes 4m13s on a cold machine and up to 87 minutes once the chassis heat-soaks —a factor of 20. A datacenter GPU removes that variable entirely, and removes it *uniformly*, whichmatters more: every task vector in this study must be produced under identical conditions or thecomparison is confounded.**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Internet **On**.This notebook is resumable. Task vectors are written as soon as each task finishes and alreadycompleted tasks are skipped on restart, so a session timeout costs only the task in flight.

## 1. Code

In [ ]:
# Repository for this project (public).REPO_URL = "https://github.com/vulpiani1948744-lab/dlai-mergeability.git"REPO_DIR = "/kaggle/working/dlai-mergeability"import os, subprocess, sysif not os.path.exists(REPO_DIR):    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)else:    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)os.chdir(REPO_DIR)sys.path.insert(0, os.path.join(REPO_DIR, "src"))print("cwd:", os.getcwd())

In [ ]:
!pip install -q timmimport torch, timmprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")print("timm ", timm.__version__)

## 2. Correctness checksThe same checks that run locally. Checks needing CIFAR-100 on disk report as skippedhere, since this runs before the dataset is downloaded.**If anything fails, stop.** Do not spend GPU hours on a broken pipeline: every bugfound in this project so far would have produced plausible-looking numbers rather thanan error.

In [ ]:
!python scripts/selftest.py

## 3. Sustained throughputMeasured, not assumed. The local estimate was wrong by 5x because it was taken from a ten-second burst.

In [ ]:
import time, torch.nn as nnfrom mergeability.models import build_backbone, TaskModeldev = torch.device("cuda")enc, spec = build_backbone("vit_tiny_patch16_224", 128, pretrained=True)model = TaskModel(enc, nn.Linear(spec.feature_dim, 20)).to(dev)opt = torch.optim.AdamW(model.encoder.parameters(), lr=1e-4)crit = nn.CrossEntropyLoss()x = torch.randn(128, 3, 128, 128, device=dev); y = torch.randint(0, 20, (128,), device=dev)for _ in range(5):    opt.zero_grad(); crit(model(x), y).backward(); opt.step()torch.cuda.synchronize()t0 = time.time(); n = 40for _ in range(n):    opt.zero_grad(); crit(model(x), y).backward(); opt.step()torch.cuda.synchronize()ips = n * 128 / (time.time() - t0)print(f"{ips:.0f} img/s  ->  {10000/ips:.0f}s per epoch  ->  {10000/ips*5/60:.1f} min per task")print(f"Benchmark A (20 tasks):        ~{10000/ips*5*20/3600:.1f} h")print(f"+ ResNet-18 control (20 more): ~{10000/ips*5*40/3600:.1f} h total")del model, enc, opt; torch.cuda.empty_cache()

## 4. Benchmark A — ViT-Tiny5 tasks x **5 regimes** x 2 seeds = **50 fine-tunings**.The five regimes are the graded similarity axis: the classes are reshuffled away fromtheir semantic grouping by a fraction alpha = 0, 0.25, 0.5, 0.75, 1, which moves measuredsemantic purity through 100 / 80 / 62 / 41 / 33 %. Every task keeps exactly 20 classes atevery level, so the training budget never varies with the axis.Already-completed tasks are skipped, so re-running this after adding a regime only trainsthe new ones.

In [ ]:
!python -u scripts/run_finetune.py --config configs/benchmark_a.yaml --device cuda

## 5. Cross-architecture control — ResNet-185 tasks x 2 regimes x 2 seeds = **20 fine-tunings**, at the two extremes of the axis only:this control asks whether the predictor ranking survives an architecture swap, not how itvaries along the axis.Identical to Benchmark A in every other respect. Watch for `froze 20 BatchNorm layers` inthe output — ResNet has BatchNorm and ViT does not, and leaving those statistics free bothcorrupted the task vectors and made fine-tuning score below its own frozen probe.

In [ ]:
!python -u scripts/run_finetune.py --config configs/benchmark_a_resnet.yaml --device cuda

## 6. Integrity checkTask vectors accumulate across sessions and sometimes across code versions, and a file written byan older version loads perfectly well while quietly poisoning every number downstream. Thishappened once already, so the check is deliberately blind to which files we suspect: it measuresevery norm and flags whatever sits far from the median.Expect **50** for ViT-Tiny (5 regimes x 5 tasks x 2 seeds) and **20** for ResNet-18 (2 regimes).

In [ ]:
!python scripts/verify_task_vectors.py checkpoints/vit_tiny_patch16_224!python scripts/verify_task_vectors.py checkpoints/resnet18

## 7. The merge experimentThe actual experiment. For every subset of tasks -- all pairs, all triples, and the full set -- andevery merging algorithm, this merges, evaluates, and writes one row carrying both the outcome(normalized accuracy) and every pre-merge signal.The signals are computed from the task vectors of the subset alone. Nothing in them can see themerged model, which is what makes them predictions rather than descriptions.Roughly 2300 merged models for ViT-Tiny and 900 for ResNet-18; budget around two hours.

In [ ]:
!python -u scripts/run_merge.py --config configs/benchmark_a.yaml --device cuda \    --out results/raw/merge_vit.csv

In [ ]:
!python -u scripts/run_merge.py --config configs/benchmark_a_resnet.yaml --device cuda \    --out results/raw/merge_resnet.csv

## 8. Package the results`merge_*.csv` is what the analysis actually needs and it is tiny; the task vectors are large andonly needed to recompute weight-space signals locally.

In [ ]:
import pandas as pd, pathlib, shutilfor name in ("merge_vit", "merge_resnet"):    f = pathlib.Path(f"results/raw/{name}.csv")    if f.exists():        d = pd.read_csv(f)        print(f"{name:14s} {len(d):6d} rows   "              f"regimes={sorted(d.regime.unique())}   methods={sorted(d.method.unique())}")out = pathlib.Path("/kaggle/working")shutil.make_archive(str(out / "results"), "zip", REPO_DIR, "results")shutil.make_archive(str(out / "task_vectors"), "zip", REPO_DIR, "checkpoints")for f in sorted(out.glob("*.zip")):    print(f"{f.name:20s} {f.stat().st_size / 1e6:8.1f} MB")